### Problema de Stokes estacionario
$$\begin{array}{l}
\begin{array}{r}
- \Delta {\bf u} + \nabla p = {\bf f} \\
\nabla \cdot{\bf u} = g \end{array} \quad\text{en }\Omega \\
\begin{array}{ll}
{\bf u}= {\bf 0} & \text{en }\Gamma_1 \cup \Gamma_2 \cup \Gamma_4  \\
{\bf u} = (1,0) & \text{en }\Gamma_3  \end{array}
\end{array}
$$
con ${\bf f} =(1,1)$, $g=0$. Condición de compatibilidad: $\int_{\partial \Omega} {\bf u}_0 \cdot \vec{n} = \int_{\Omega} g$

In [ ]:
%reset -f
import mfem.ser as mfem
import numpy as np

### Malla

In [ ]:
mesh = mfem.Mesh.MakeCartesian2D(30,30,mfem.Geometry.SQUARE)
dim = mesh.Dimension()
mesh.bdr_attributes.ToList()

#### Espacios
$H^2$ para la velocidad, $H^1$ para la presión

In [ ]:
order = 2
h2 = mfem.H1_FECollection(order, dim)
h1 = mfem.H1_FECollection(order-1, dim)

U_space = mfem.FiniteElementSpace(mesh, h2, dim)
P_space = mfem.FiniteElementSpace(mesh, h1)

#### Dimensiones de los espacios y configuración de los bloques

In [ ]:
dimR = U_space.GetVSize()
dimW = P_space.GetVSize()

block_offsets = mfem.intArray([0, dimR, dimW])
block_offsets.PartialSum()
block_offsets.ToList()

x = mfem.BlockVector(block_offsets)
x.Assign(0.)
rhs = mfem.BlockVector(block_offsets)

### Condición frontera

In [ ]:
# Todas las fronteras son Dirichlet
boundary_dofs = mfem.intArray([1,1,1,1])

# Condición Dirichlet no nula
ess_bdr = mfem.intArray([0,0,1,0])

#### Imponemos valores en la frontera de x

In [ ]:
ux = mfem.GridFunction(U_space)
ux.MakeRef(U_space,x.GetBlock(0),0)

f1 = mfem.VectorConstantCoefficient([1.,0.])
ux.ProjectBdrCoefficient(f1,ess_bdr)

### Formulación variacional
$$\begin{array}{c}
\displaystyle \int_\Omega \nabla{\bf u}:\nabla {\bf v} - \int_\Omega  p(\nabla\cdot {\bf v}) = \int_\Omega {\bf f}\cdot {\bf v} \\ 
-\displaystyle \int_\Omega (\nabla \cdot{\bf u})q = 0
\end{array}$$
con ${\bf v} \in H^2(\Omega)$, y $q\in H^1(\Omega)$

##### Matricialmente:
$$\begin{array}{l}
M \longrightarrow \displaystyle \int_\Omega \nabla{\bf u}:\nabla {\bf v} \\
B \longrightarrow \displaystyle -\int_\Omega \nabla\cdot {\bf u} \cdot q \longrightarrow 
B^T \longrightarrow \displaystyle -\int_\Omega p(\nabla \cdot {\bf v}) \end{array}$$

$$\begin{bmatrix} M & B \\ B^T & 0\end{bmatrix} \begin{bmatrix} U \\ P \end{bmatrix} = 
\begin{bmatrix} F \\ -G \end{bmatrix} $$

#### Segundos miembros

In [ ]:
class fFunc(mfem.VectorPyCoefficient):
    def EvalValue(self, x):
        return [1. , 1.]
        
fcoeff = fFunc(dim)

In [ ]:
fform = mfem.LinearForm()
fform.Update(U_space, rhs.GetBlock(0), 0)
fform.AddDomainIntegrator(mfem.VectorDomainLFIntegrator(fcoeff))
fform.Assemble()

gform = mfem.LinearForm()
gform.Update(P_space, rhs.GetBlock(1), 0)
gform.Assemble()

#### Formas bilineales
Como ahora hay condiciones Dirichlet, debemos fijar los valores en los nodos correspondientes con `EliminateEssentialBC`

In [ ]:
mVarf = mfem.BilinearForm(U_space)
mVarf.AddDomainIntegrator(mfem.VectorDiffusionIntegrator())
mVarf.Assemble()
mVarf.EliminateEssentialBC(boundary_dofs,ux,rhs.GetBlock(0))
mVarf.Finalize()

In [ ]:
bVarf = mfem.MixedBilinearForm(U_space, P_space)
bVarf.AddDomainIntegrator(mfem.VectorDivergenceIntegrator(mfem.ConstantCoefficient(-1.)))
bVarf.Assemble()
bVarf.EliminateTrialEssentialBC(boundary_dofs, ux, rhs.GetBlock(1))
bVarf.Finalize()

### Matriz del sistema

In [ ]:
stokesOp = mfem.BlockOperator(block_offsets)

M = mVarf.SpMat()
B = bVarf.SpMat()
Bt = mfem.TransposeOperator(B)

stokesOp.SetBlock(0, 0, M)
stokesOp.SetBlock(0, 1, Bt)
stokesOp.SetBlock(1, 0, B)

### Resolución

In [ ]:
solver = mfem.MINRESSolver()

solver.SetAbsTol(0.)
solver.SetRelTol(1.e-5)
solver.SetMaxIter(5000)

solver.SetOperator(stokesOp)
solver.SetPrintLevel(0)

solver.Mult(rhs, x)


if solver.GetConverged():
    print("MINRES convergió en " + str(solver.GetNumIterations()) +
          " iteraciones, con residuo " + "{:g}".format(solver.GetFinalNorm()))
else:
    print("MINRES no ha convergido tras " + str(solver.GetNumIterations()) +
        " iteraciones. Residuo:" + "{:g}".format(solver.GetFinalNorm()))

#### Asignamos el vector solución a funciones para dibujar

In [ ]:
u = mfem.GridFunction()
p = mfem.GridFunction()
u.MakeRef(U_space, x.GetBlock(0), 0)
p.MakeRef(P_space, x.GetBlock(1), 0)

In [ ]:
sout_u = mfem.socketstream("localhost", 19916)
sout_u.precision(8)
sout_u << "solution\n" << mesh << u
sout_u << "window_title 'Velocity'"
sout_u << "keys 'Rvvvv'\n" ;
sout_u.close()

In [ ]:
sout_p = mfem.socketstream("localhost", 19916)
sout_p.precision(8)
sout_p << "solution\n" << mesh << p 
sout_p << "window_title 'Pressure'"
sout_p.close()